# 指针

> **开始前请注意：在线输入限制**
> 当前在线环境中的 `scanf`、`getchar`、`fgets(..., stdin)` 无法交互读取键盘输入。在线实验请修改变量的初值，或使用 `sscanf` 从字符串读取；键盘输入练习请在本地 GCC / Clang 中运行完整 C 程序。

**本章目标**：解释数组到指针的转换；识别空指针和悬空指针；正确管理动态内存。

**学习方法**：先预测输出，再运行验证；每次只修改一个条件，最后用自己的话解释变化。修改函数或类型定义后，请重启内核并从头运行。

**本章学习路线**：

- **基础必做**：地址与解引用、指针与数组、指针参数先修小课、空指针与生命周期、malloc/free、贯穿案例5。
- **综合应用**：指针作图、本地人数输入与动态数组统计。
- **选学**：calloc/realloc、二重指针、从字符串装入动态数组。选学内容不作为首次学习的前置要求。

## 1. 什么是指针？
- 指针是 **存储变量地址的变量**。
- 本章使用普通对象变量的地址；指针用于间接访问仍然存在的对象。

### 地址运算符 `&`
- `&a` 表示变量 `a` 的地址。

## 2. 定义指针

类型 *指针变量名;

In [ ]:
#include <stdio.h>
{
    int a = 10;
    int *p = &a; // 先初始化，再使用；未初始化指针的值不确定，不能读取或解引用
    printf("通过指针访问 a 的值: %d\n", *p);
}

## 3. 指针与数组

数组是连续存放元素的对象，指针是保存地址的对象，二者类型不同。

数组表达式在多数场景下会转换为指向首元素的指针，例如 `int *p = arr;`。但 `sizeof arr` 取得整个数组的大小，`&arr` 则是指向整个数组的指针。这些场景不能简单理解为“数组名就是指针常量”。

`p + 1` 指向下一个元素，前进的字节数由元素类型决定。只能在同一数组及其尾后位置内进行相应指针运算；尾后指针可以作为终止标记，但不能解引用。

In [ ]:
{
    int arr[3] = {10, 20, 30};
    int *p = arr; // 等价于 p = &arr[0];
    
    printf("%d\n", *p);      // 10
    printf("%d\n", *(p+1));  // 20
    printf("%d\n", *(p+2));  // 30
    //指针遍历数组
    for (int i = 0; i < 3; i++) {
        printf("%d ", *(p+i));
    }
}

## 4. 指针与函数参数

**先修小课**：`void swap(int *x, int *y)` 定义一个不返回值的函数，两个参数均为指向 int 的指针；`swap(&a, &b)` 是调用。函数的完整知识在第 6 章展开。

C 的参数传递都是值传递，这里复制的是地址。通过复制的地址访问对象，能够修改调用者的变量；重新给形参指针本身赋值则不会改写调用者的指针。

下面的 swap 是专门演示“通过指针修改对象”的操作，前提是两个指针都指向有效的 int 对象。

In [ ]:
//示例：交换两个数
void swap(int *x, int *y) {
    int temp = *x;
    *x = *y;
    *y = temp;
}

{
    int a = 5, b = 10;
    swap(&a, &b);
    printf("a=%d, b=%d\n", a, b); // a=10, b=5
}

## 5. 指针与字符串

`char str[] = "Hello";` 创建可修改的数组；`const char *text = "Hello";` 指向字符串字面量，应只读访问。字符串字面量本身具有数组类型，在多数表达式中转换为首字符指针；不能修改字面量内容。

In [ ]:
{
    char str[] = "Hello";
    char *p = str;
    
    printf("%s\n", p);   // Hello
    printf("%c\n", *(p+1)); // e
}

## 6. 指针与结构体

可以使用指针访问结构体，常用 -> 运算符。

In [ ]:
struct Student {
    char name[20];
    int age;
};

{
    struct Student s = {"Alice", 18};
    struct Student *p = &s;

    printf("姓名: %s\n", p->name);
    printf("年龄: %d\n", p->age);
}

## 7. 内存分配与释放

普通局部对象的自动存储期通常对应栈实现；动态分配的存储通常称为堆。C 标准规定对象的存储期，不要求某一种物理布局。

`<stdlib.h>` 中的函数：

| 函数 | 要点 |
| --- | --- |
| `malloc(bytes)` | 分配未初始化存储，成功后仍应先写后读 |
| `calloc(count, size)` | 将所有位初始化为0；这里用于整数数组 |
| `realloc(ptr, bytes)` | 本章只传正的非零大小；失败返回 NULL，原分配仍有效 |
| `free(ptr)` | 释放动态分配；同一块存储只能释放一次，`free(NULL)` 无操作 |

C 中不需要强制转换 `malloc` 的返回值。先检查分配是否成功，再访问元素。用临时指针接收 `realloc`；成功后可能迁移内存，原地址及其别名都不能继续使用。`free` 不会自动清空任何指针；给其中一个指针赋 NULL，也不能修复其他悬空别名。

### 小实验 A：malloc 后先写再读

只观察一件事：分配成功并不代表元素已有可读取的初值。每条路径都要有明确的释放责任。

In [ ]:
#include <stdlib.h>
{
    int *values = malloc(3 * sizeof *values);
    if (values == NULL) { printf("malloc 失败\n"); }
    else {
        values[0] = 85;
        printf("第一个成绩=%d\n", values[0]);
        free(values);
    }
}

### 选学·小实验 B：calloc 的整数初值

这次读取初始化后的整数元素。不要由此推论所有类型的全零位模式都具有同样含义。

In [ ]:
{
    int *values = calloc(3, sizeof *values);
    if (values == NULL) { printf("calloc 失败\n"); }
    else { printf("整数初值=%d %d %d\n", values[0], values[1], values[2]); free(values); }
}

### 选学·小实验 C：扩容后哪些值保留？

扩容成功会保留原有元素；新增部分先赋值再读取。失败时释放仍有效的原分配。成功后只使用返回的新指针，不比较或读取旧地址。

In [ ]:
{
    int *values = malloc(2 * sizeof *values);
    if (values == NULL) { printf("malloc 失败\n"); }
    else {
        values[0] = 85;
        values[1] = 90;
        int *expanded = realloc(values, 3 * sizeof *values);
        if (expanded == NULL) { printf("realloc 失败\n"); free(values); }
        else {
            values = expanded;
            values[2] = 58;
            printf("扩容后=%d %d %d\n", values[0], values[1], values[2]);
            free(values);
        }
    }
}

## 实验 1：看地址、看大小、看步长

**预测**下面的数组元素数与指针差值。记录两个 sizeof 的结果，但不要要求它们在所有平台上都不同。修改数组为 4 个元素后再比较。

地址图（每个方框代表一个 int）：

```text
arr: [10] [20] [30]   尾后位置
       ^    ^           ^
       p   p+1         arr+3（不可解引用）
```

`(p + 1) - p` 计算的是元素数，输出格式 `%td` 对应 `ptrdiff_t`。

In [ ]:
{
    int arr[3] = {10, 20, 30};
    int *p = arr;
    printf("数组字节=%zu 指针字节=%zu\n", sizeof arr, sizeof p);
    printf("元素数=%zu 步长=%td\n", sizeof arr / sizeof arr[0], (p + 1) - p);
    printf("首元素相同=%d 下一个元素=%d\n", p == &arr[0], *(p + 1));
}

## 实验 2：空指针与对象生命周期

下面只演示检查空指针，不执行无效解引用。**修改**：让 p 指向一个仍然存在的 int，再输出它的值。

以下行为只讨论、不运行：解引用 NULL、返回局部变量地址后再访问、free 后读取、越界访问。它们属于未定义行为，不保证报错，也不保证每次结果相同。

In [ ]:
{
    const int *p = NULL;
    if (p == NULL) { printf("没有可读取的对象\n"); }
    else { printf("%d\n", *p); }
}

## 贯穿案例 5：按人数分配、赋值、统计、释放

在线用 count 模拟运行时得到的人数，范围为1..100。先为每人成绩赋初值60，再把前三名设为85、90、58；本地实践会用键盘输入替换这些模拟数据。

先预测 count=3 时的平均分。随后改为1、4、0：人数为4时新增成员保持60分；人数为0时应被拒绝。注意分配与释放出现的位置，每条成功分配的路径都应释放一次。

本例只练习内存与数组访问。文本解析放在本章末尾的选学部分。

In [ ]:
{
    size_t count = 3;
    if (count == 0 || count > 100) { printf("人数必须为1..100\n"); }
    else {
        int *scores = malloc(count * sizeof *scores);
        if (scores == NULL) { printf("成绩空间分配失败\n"); }
        else {
            for (size_t i = 0; i < count; ++i) { scores[i] = 60; }
            scores[0] = 85; // 前面已经确认至少有1人。
            if (count > 1) { scores[1] = 90; }
            if (count > 2) { scores[2] = 58; }
            int total = 0;
            for (size_t i = 0; i < count; ++i) { total += scores[i]; }
            printf("人数=%zu 平均分=%.2f\n", count, (double)total / count);
            free(scores);
        }
    }
}

## 指针作图练习：改地址还是改对象？

在纸上画出 value、other、p、q 的关系，再运行。去掉或恢复 `p = &other`，每次重新运行整个单元。解释为什么修改 p 不会自动改变 q。

`const int *p` 限制通过 p 修改所指对象；不意味着 p 本身不能改指向。

In [ ]:
{
    int value = 85, other = 90;
    int *p = &value;
    int *q = p;
    *p = 60;
    p = &other;
    printf("value=%d *p=%d *q=%d\n", value, *p, *q);
}

## 分层练习

### 1. 读程序
`int a[3]; int *p = a;` 中 sizeof a 与 sizeof p 分别衡量什么？

<details><summary>提示：先自己尝试</summary>

数组与指针具有不同类型。

</details>

<details><summary>参考思路与自查</summary>

前者为整个数组的字节数，后者为指针对象的字节数；不能用 sizeof p 推断数组长度。

</details>

### 2. 改错
`int *p; *p = 10;` 错在哪里？给出不分配动态内存的修正。

<details><summary>提示：先自己尝试</summary>

需要一个仍然存在的 int 对象。

</details>

<details><summary>参考思路与自查</summary>

例如 `int value = 0; int *p = &value; *p = 10;`。不能通过尝试随机地址来修复。

</details>

### 3. 编程
用只读指针遍历 `{85,90,58}` 并求和，预期 233。

<details><summary>提示：先自己尝试</summary>

使用 `const int *p`，停止于数组尾后位置；只在有效元素上解引用。

</details>

<details><summary>参考思路与自查</summary>

可以用下标或 p 逐项前进；尾后指针只用于比较，不能读取。

</details>

### 4. 应用
为什么不能直接写 `p = realloc(p, larger_size);`？若 q 也指向原分配，只给 p 赋 NULL 是否足够？

<details><summary>提示：先自己尝试</summary>

分别考虑失败、成功迁移，以及其他别名。

</details>

<details><summary>参考思路与自查</summary>

失败时会丢失原地址；用临时指针。成功后重建需要的别名，释放后任何别名都不能解引用。

</details>

## 本地实践：完整 C17 程序

将下面代码保存为 `chapter05.c`，执行 `cc -std=c17 -Wall -Wextra -Wpedantic chapter05.c -o chapter05`，再运行 `./chapter05`（Windows 使用 `chapter05.exe`）。此代码展示完整程序结构，不作为 Notebook 单元执行。

```c
#include <stdio.h>
#include <stdlib.h>

int main(void) {
    int count;
    if (scanf("%d", &count) != 1 || count < 1 || count > 100) {
        printf("人数必须在1到100之间\n");
        return 1;
    }
    int *scores = malloc((size_t)count * sizeof *scores);
    if (scores == NULL) {
        printf("成绩空间分配失败\n");
        return 1;
    }
    int total = 0;
    for (int i = 0; i < count; ++i) {
        if (scanf("%d", &scores[i]) != 1 || scores[i] < 0 || scores[i] > 100) {
            printf("无效成绩\n");
            free(scores);
            return 1;
        }
        total += scores[i];
    }
    printf("人数=%d 平均分=%.2f\n", count, (double)total / count);
    free(scores);
    return 0;
}
```

默认样例的标准输出：

```text
人数=3 平均分=77.67
```

本地输入示例：第一行 `3`，第二行 `85 90 58`。先限制人数，再分配空间；每次提前退出都释放已获得的存储。

## 选学：指向指针的指针（二重指针）

C 语言允许指针指向另一个指针。

先掌握一个指针指向普通对象即可；此处只观察层级关系，后续遇到需要修改指针本身的接口时再深入。

In [ ]:
{
    int a = 10;
    int *p = &a;
    int **pp = &p;
    
    printf("%d\n", **pp); // 10
}

## 选学：从字符串装入动态数组

本地程序从键盘读取人数；在线用 count 模拟运行时得到的人数，并从有界的样例字符串中读取成绩。这部分同时涉及文本解析与指针前移，完成基础分配实验后再阅读。

默认 count=3、文本为 `"85 90 58"`，与前后章节使用相同成绩。先预测平均分，再把 count 改为1；随后改为4，观察数据不足时如何报告错误。人数只允许1..100，样例数字只使用 -1..101；不将此演示当作任意外部文本的通用解析器。

In [ ]:
{
    size_t count = 3;
    const char *cursor = "85 90 58";
    if (count == 0 || count > 100) { printf("人数必须为1..100\n"); }
    else {
        int *scores = malloc(count * sizeof *scores);
        if (scores == NULL) { printf("成绩空间分配失败\n"); }
        else {
            int valid = 1, total = 0;
            for (size_t i = 0; i < count; ++i) {
                int consumed = 0;
                if (sscanf(cursor, "%d%n", &scores[i], &consumed) != 1 || scores[i] < 0 || scores[i] > 100) {
                    valid = 0;
                    break;
                }
                cursor += consumed;
                total += scores[i];
            }
            if (valid) { printf("人数=%zu 平均分=%.2f\n", count, (double)total / count); }
            else { printf("成绩不足或不在0..100范围\n"); }
            free(scores);
        }
    }
}

`%n` 把本次已消耗的字符数写入 consumed，它本身不计入返回的转换项数。cursor 前移后读取下一项；`||` 的短路保证转换失败时不读取未初始化的成绩。只读取 count 项，额外文本不在本实验的校验范围。

## 小结

数组不是指针对象；理解转换发生的场景。解引用前确认对象存在、地址有效、访问未越界；动态内存需要清楚的分配与释放责任。